# Disentangle Spaghetti

Run from the task code directory. The notebook mirrors the production Python script in structured sections.


## Setup and Parameters


In [ ]:
__file__ = 'build_disentangle_spaghetti.py'
#!/usr/bin/env python3
"""
Disentangle model heterogeneity in MMB policy-transmission results.

This script runs a deeper analysis layer beyond the paper's baseline regressions:
1) Rule-vs-model variance decomposition (fixed-effects OLS)
2) Cross-rule rank stability by outcome
3) IRF-shape archetype clustering and attribute contrasts
4) Nonlinear benchmark vs linear benchmark
5) Outlier sensitivity checks

Inputs:
  - ../input/MMB_reg_format.dta
  - ../input/MMB_IRF_format_full.dta

Outputs:
  - ../output/disentangle_spaghetti/

Run:
  make
"""


from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm
import statsmodels.formula.api as smf
import statsmodels.api as sm

from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler


OUTCOMES = ["IScurve20", "infl_per_rr20", "sacratio20", "y_timing_max", "piq_timing_max"]
RULE_PAIRS = [("Taylor", "Inertial_Taylor"), ("Taylor", "Growth"), ("Inertial_Taylor", "Growth")]
CORE_FEATURES = [
    "rule_itr",
    "rule_g",
    "estimated",
    "stky_wg",
    "wg_ndx",
    "pr_ndx",
    "hh_demand",
    "bank",
    "firm_bs",
    "labor_frict",
    "open",
    "cb_authors_ext",
    "ln_neq",
    "vint_mid",
    "vint_late",
]

ELASTICITY_OUTCOMES = ["IScurve20", "infl_per_rr20", "sacratio20"]
TIMING_OUTCOMES = ["y_timing_max", "piq_timing_max"]

DRIVER_CONTROL_FEATURES = ["rule_itr", "rule_g", "estimated"]
DRIVER_NOMINAL_FEATURES = ["sw_x_idx", "sw_x_noidx", "pr_ndx"]
DRIVER_REAL_FEATURES = [
    "hh_demand",
    "bank",
    "firm_bs",
    "labor_frict",
    "open",
    "hh_demand_x_bank",
    "firm_bs_x_bank",
    "hh_demand_x_firm_bs",
    "hh_demand_x_labor_frict",
    "firm_bs_x_labor_frict",
    "bank_x_labor_frict",
    "hh_demand_x_open",
    "firm_bs_x_open",
    "bank_x_open",
    "labor_frict_x_open",
]
DRIVER_NONMODEL_FEATURES = ["cb_authors_ext", "ln_neq", "vint_mid", "vint_late", "cb_x_late"]
DRIVER_ALL_FEATURES = (
    DRIVER_CONTROL_FEATURES
    + DRIVER_NOMINAL_FEATURES
    + DRIVER_REAL_FEATURES
    + DRIVER_NONMODEL_FEATURES
)


TASK_DIR = Path(__file__).resolve().parents[1]
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"




## Reused Helper Steps


### load_data


In [ ]:
def load_data():
    reg = pd.read_stata(INPUT_DIR / "MMB_reg_format.dta")
    # Keep paper-like timing sample and non-missing model/rule IDs.
    reg = reg[(reg["y_timing_max"] != 99) & (reg["piq_timing_max"] != 99)].copy()
    reg = reg.dropna(subset=["model", "rule"]).copy()

    irf = pd.read_stata(INPUT_DIR / "MMB_IRF_format_full.dta")
    irf = irf[(irf["period"] <= 20)].copy()
    irf = irf.dropna(subset=["model", "rule"]).copy()

    # Restrict IRF panel to model-rule keys that survive the regression sample filter.
    valid_keys = set(reg["model"].astype(str) + "||" + reg["rule"].astype(str))
    irf["key"] = irf["model"].astype(str) + "||" + irf["rule"].astype(str)
    irf = irf[irf["key"].isin(valid_keys)].copy()

    return reg, irf




### variance_decomposition


In [ ]:
def variance_decomposition(reg):
    rows = []
    for outcome in OUTCOMES:
        d = reg[["model", "rule", outcome]].dropna().copy()
        if d.empty:
            continue
        m_rule = smf.ols(f"{outcome} ~ C(rule)", data=d).fit()
        m_model = smf.ols(f"{outcome} ~ C(model)", data=d).fit()
        m_both = smf.ols(f"{outcome} ~ C(rule) + C(model)", data=d).fit()
        rows.append(
            {
                "outcome": outcome,
                "n_obs": len(d),
                "r2_rule_only": m_rule.rsquared,
                "r2_model_only": m_model.rsquared,
                "r2_rule_plus_model": m_both.rsquared,
                "inc_rule_given_model": m_both.rsquared - m_model.rsquared,
                "inc_model_given_rule": m_both.rsquared - m_rule.rsquared,
            }
        )
    return pd.DataFrame(rows).sort_values("outcome").reset_index(drop=True)




### rule_rank_stability


In [ ]:
def rule_rank_stability(reg):
    rows = []
    for outcome in OUTCOMES:
        piv = reg.pivot_table(index="model", columns="rule", values=outcome)
        for a, b in RULE_PAIRS:
            if a not in piv.columns or b not in piv.columns:
                continue
            d = piv[[a, b]].dropna()
            if len(d) < 10:
                continue
            sp = stats.spearmanr(d[a], d[b])
            pe = stats.pearsonr(d[a], d[b])
            rows.append(
                {
                    "outcome": outcome,
                    "rule_a": a,
                    "rule_b": b,
                    "n_models": len(d),
                    "spearman_rho": sp.statistic,
                    "spearman_p": sp.pvalue,
                    "pearson_r": pe.statistic,
                    "pearson_p": pe.pvalue,
                }
            )
    return pd.DataFrame(rows).sort_values(["outcome", "rule_a", "rule_b"]).reset_index(drop=True)




### within_model_rule_contrasts


In [ ]:
def within_model_rule_contrasts(reg):
    rows = []
    for outcome in OUTCOMES:
        piv = reg.pivot_table(index=["model", "estimated"], columns="rule", values=outcome)
        for a, b in RULE_PAIRS:
            if a not in piv.columns or b not in piv.columns:
                continue
            d = piv[[a, b]].dropna().copy()
            if len(d) < 10:
                continue
            d["delta"] = d[b] - d[a]
            rows.append(
                {
                    "outcome": outcome,
                    "contrast": f"{b} - {a}",
                    "group": "all_models",
                    "n_models": len(d),
                    "mean_delta": d["delta"].mean(),
                    "median_delta": d["delta"].median(),
                    "t_test_p": stats.ttest_1samp(d["delta"], 0.0, nan_policy="omit").pvalue,
                }
            )

            for est_flag, grp_name in [(0, "calibrated_models"), (1, "estimated_models")]:
                dg = d.loc[d.index.get_level_values("estimated") == est_flag, "delta"]
                if len(dg) < 5:
                    continue
                rows.append(
                    {
                        "outcome": outcome,
                        "contrast": f"{b} - {a}",
                        "group": grp_name,
                        "n_models": len(dg),
                        "mean_delta": dg.mean(),
                        "median_delta": dg.median(),
                        "t_test_p": stats.ttest_1samp(dg, 0.0, nan_policy="omit").pvalue,
                    }
                )

            d_cal = d.loc[d.index.get_level_values("estimated") == 0, "delta"]
            d_est = d.loc[d.index.get_level_values("estimated") == 1, "delta"]
            if len(d_cal) >= 5 and len(d_est) >= 5:
                rows.append(
                    {
                        "outcome": outcome,
                        "contrast": f"{b} - {a}",
                        "group": "estimated_minus_calibrated",
                        "n_models": min(len(d_cal), len(d_est)),
                        "mean_delta": d_est.mean() - d_cal.mean(),
                        "median_delta": d_est.median() - d_cal.median(),
                        "t_test_p": stats.ttest_ind(d_est, d_cal, equal_var=False, nan_policy="omit").pvalue,
                    }
                )

    return pd.DataFrame(rows).sort_values(["outcome", "contrast", "group"]).reset_index(drop=True)




### outlier_table


In [ ]:
def outlier_table(reg):
    d = reg[["model", "rule"] + OUTCOMES].copy()
    for c in OUTCOMES:
        q1, q99 = d[c].quantile([0.01, 0.99])
        d[f"{c}_out_1_99"] = (d[c] < q1) | (d[c] > q99)
    d["n_outlier_flags"] = d[[f"{c}_out_1_99" for c in OUTCOMES]].sum(axis=1)
    return d.loc[d["n_outlier_flags"] > 0].sort_values("n_outlier_flags", ascending=False).reset_index(drop=True)




### build_irf_shape_features


In [ ]:
def build_irf_shape_features(irf):
    rows = []
    for key, g in irf.groupby("key"):
        g = g.sort_values("period")
        row = {"key": key, "model": g["model"].iloc[0], "rule": g["rule"].iloc[0]}
        for v in ["y", "piq", "irate", "rrate"]:
            vals = g[v].to_numpy(dtype=float)
            row[f"{v}_cum20"] = np.nansum(vals)
            row[f"{v}_peak"] = np.nanmax(vals)
            row[f"{v}_trough"] = np.nanmin(vals)
            row[f"{v}_std"] = np.nanstd(vals)
            row[f"{v}_tpeak"] = int(g.iloc[np.nanargmax(vals)]["period"])
            row[f"{v}_ttrough"] = int(g.iloc[np.nanargmin(vals)]["period"])
        rows.append(row)
    return pd.DataFrame(rows)




### cluster_archetypes


In [ ]:
def cluster_archetypes(reg, irf):
    shape = build_irf_shape_features(irf)
    x = shape[[c for c in shape.columns if c not in ["key", "model", "rule"]]].copy()

    # Robust winsorization and scaling.
    for c in x.columns:
        lo, hi = x[c].quantile([0.02, 0.98])
        x[c] = x[c].clip(lo, hi)
    x_scaled = RobustScaler().fit_transform(x)

    best = None
    for k in range(2, 7):
        km = KMeans(n_clusters=k, random_state=42, n_init=50).fit(x_scaled)
        s = silhouette_score(x_scaled, km.labels_)
        if best is None or s > best[0]:
            best = (s, k, km.labels_)

    shape["cluster_raw"] = best[2]
    shape["k_selected"] = best[1]
    shape["silhouette"] = best[0]

    merged = shape.merge(
        reg[
            [
                "model",
                "rule",
                "IScurve20",
                "infl_per_rr20",
                "sacratio20",
                "y_timing_max",
                "piq_timing_max",
                "estimated",
                "cb_authors_ext",
                "hh_demand",
                "firm_bs",
                "bank",
                "labor_frict",
                "open",
                "rule_itr",
                "rule_g",
            ]
        ],
        on=["model", "rule"],
        how="left",
    )

    # Orient binary cluster label for k=2: slow cluster has higher y_timing mean.
    if best[1] == 2:
        means = merged.groupby("cluster_raw")["y_timing_max"].mean()
        slow_cluster = int(means.idxmax())
        merged["cluster_slow"] = (merged["cluster_raw"] == slow_cluster).astype(int)
    else:
        merged["cluster_slow"] = np.nan

    cluster_outcomes = (
        merged.groupby("cluster_raw")[["IScurve20", "infl_per_rr20", "sacratio20", "y_timing_max", "piq_timing_max"]]
        .mean()
        .reset_index()
    )
    cluster_prevalence = (
        merged.groupby("cluster_raw")[["estimated", "cb_authors_ext", "hh_demand", "firm_bs", "bank", "labor_frict", "open", "rule_itr", "rule_g"]]
        .mean()
        .reset_index()
    )

    return shape, merged, cluster_outcomes, cluster_prevalence




### cluster_stability


In [ ]:
def cluster_stability(shape_features, cluster_outcomes):
    d = shape_features[["model", "rule", "cluster_raw"]].copy()
    cluster_order = (
        cluster_outcomes[["cluster_raw", "y_timing_max"]]
        .sort_values("y_timing_max")
        .reset_index(drop=True)
    )
    if len(cluster_order) >= 2:
        fast_cluster = int(cluster_order.loc[0, "cluster_raw"])
        slow_cluster = int(cluster_order.loc[len(cluster_order) - 1, "cluster_raw"])
    else:
        fast_cluster = slow_cluster = int(cluster_order.loc[0, "cluster_raw"])

    model_cluster_counts = d.groupby(["model", "cluster_raw"]).size().unstack(fill_value=0)
    share = model_cluster_counts.div(model_cluster_counts.sum(axis=1), axis=0)
    dominant_cluster = share.idxmax(axis=1)
    dominant_share = share.max(axis=1)
    n_clusters_seen = (model_cluster_counts > 0).sum(axis=1)
    n_rules = model_cluster_counts.sum(axis=1)

    model_table = pd.DataFrame(
        {
            "model": share.index,
            "n_rules_present": n_rules.values,
            "n_clusters_seen": n_clusters_seen.values,
            "dominant_cluster": dominant_cluster.values,
            "dominant_share": dominant_share.values,
        }
    )

    def classify(r):
        if r["n_clusters_seen"] == 1 and int(r["dominant_cluster"]) == fast_cluster:
            return "stable_fast_high_power"
        if r["n_clusters_seen"] == 1 and int(r["dominant_cluster"]) == slow_cluster:
            return "stable_slow_moderate"
        if r["n_clusters_seen"] == 1:
            return "stable_other_cluster"
        return "rule_switcher"

    model_table["stability_class"] = model_table.apply(classify, axis=1)
    model_table = model_table.sort_values(["stability_class", "model"]).reset_index(drop=True)

    summary = (
        model_table.groupby("stability_class")
        .agg(n_models=("model", "size"), avg_dominant_share=("dominant_share", "mean"))
        .reset_index()
        .sort_values("n_models", ascending=False)
    )
    return model_table, summary




### build_driver_features


In [ ]:
def build_driver_features(d):
    x = pd.DataFrame(index=d.index)
    x["rule_itr"] = d["rule_itr"].astype(float)
    x["rule_g"] = d["rule_g"].astype(float)
    x["estimated"] = d["estimated"].astype(float)
    x["sw_x_idx"] = d["stky_wg"].astype(float) * d["wg_ndx"].astype(float)
    x["sw_x_noidx"] = d["stky_wg"].astype(float) * (1.0 - d["wg_ndx"].astype(float))
    x["pr_ndx"] = d["pr_ndx"].astype(float)
    x["hh_demand"] = d["hh_demand"].astype(float)
    x["bank"] = d["bank"].astype(float)
    x["firm_bs"] = d["firm_bs"].astype(float)
    x["labor_frict"] = d["labor_frict"].astype(float)
    x["open"] = d["open"].astype(float)
    x["hh_demand_x_bank"] = x["hh_demand"] * x["bank"]
    x["firm_bs_x_bank"] = x["bank"] * x["firm_bs"]
    x["hh_demand_x_firm_bs"] = x["firm_bs"] * x["hh_demand"]
    x["hh_demand_x_labor_frict"] = x["hh_demand"] * x["labor_frict"]
    x["firm_bs_x_labor_frict"] = x["firm_bs"] * x["labor_frict"]
    x["bank_x_labor_frict"] = x["bank"] * x["labor_frict"]
    x["hh_demand_x_open"] = x["hh_demand"] * x["open"]
    x["firm_bs_x_open"] = x["firm_bs"] * x["open"]
    x["bank_x_open"] = x["bank"] * x["open"]
    x["labor_frict_x_open"] = x["labor_frict"] * x["open"]
    x["cb_authors_ext"] = d["cb_authors_ext"].astype(float)
    x["ln_neq"] = d["ln_neq"].astype(float)
    x["vint_mid"] = d["vint_mid"].astype(float)
    x["vint_late"] = d["vint_late"].astype(float)
    x["cb_x_late"] = x["cb_authors_ext"] * x["vint_late"]
    x = x.replace([np.inf, -np.inf], np.nan)
    for col in x.columns:
        med = x[col].median()
        x[col] = x[col].fillna(med if pd.notna(med) else 0.0)
    return x




### interpret_effect


In [ ]:
def interpret_effect(outcome, coef):
    if outcome in ["IScurve20", "infl_per_rr20"]:
        return "stronger policy power (larger absolute slope)" if coef < 0 else "weaker policy power (smaller absolute slope)"
    if outcome == "sacratio20":
        return "higher output cost of disinflation" if coef > 0 else "lower output cost of disinflation"
    if outcome in TIMING_OUTCOMES:
        return "longer lag to peak effect" if coef > 0 else "shorter lag to peak effect"
    return ""


def drop_collinear_columns(x):
    keep = []
    rank = 0
    for col in x.columns:
        trial = x[keep + [col]]
        trial_rank = np.linalg.matrix_rank(trial.to_numpy(dtype=float))
        if trial_rank > rank:
            keep.append(col)
            rank = trial_rank
    return x[keep]



### attribute_driver_models


In [ ]:
def attribute_driver_models(reg):
    x_panel = build_driver_features(reg)
    x_panel = sm.add_constant(x_panel, has_constant="add")

    panel_rows = []
    for outcome in OUTCOMES:
        y = pd.to_numeric(reg[outcome], errors="coerce")
        m = y.notna()
        xi = drop_collinear_columns(x_panel.loc[m].copy())
        yi = y.loc[m].astype(float)
        if outcome in ELASTICITY_OUTCOMES:
            fit = sm.RLM(yi, xi, M=sm.robust.norms.HuberT()).fit(maxiter=200)
            pvals = 2.0 * norm.sf(np.abs(fit.params / fit.bse))
            pvals = pd.Series(pvals, index=fit.params.index)
            params = fit.params
            ses = fit.bse
        else:
            fit = sm.GLM(yi, xi, family=sm.families.NegativeBinomial(alpha=1.0)).fit(
                cov_type="cluster",
                cov_kwds={"groups": reg.loc[m, "model"]},
                maxiter=200,
            )
            params = fit.params
            ses = fit.bse
            pvals = fit.pvalues

        for f in params.index:
            panel_rows.append(
                {
                    "sample": "panel_with_rule_controls",
                    "outcome": outcome,
                    "feature": f,
                    "coef": float(params[f]),
                    "se": float(ses[f]),
                    "p_value": float(pvals[f]),
                    "interpretation": interpret_effect(outcome, float(params[f])),
                }
            )

    panel_coef = pd.DataFrame(panel_rows)
    panel_sig = panel_coef[
        (panel_coef["feature"] != "const")
        & (~panel_coef["feature"].isin(DRIVER_CONTROL_FEATURES))
        & (panel_coef["p_value"] <= 0.10)
    ].copy()
    panel_sig = panel_sig.sort_values(["outcome", "p_value"]).reset_index(drop=True)

    # Model-level check: average outcomes across rules to isolate cross-model differences.
    reg_model = reg.groupby("model", as_index=False).mean(numeric_only=True)
    x_model = build_driver_features(reg_model).drop(columns=["rule_itr", "rule_g"], errors="ignore")
    x_model = sm.add_constant(x_model, has_constant="add")

    model_rows = []
    for outcome in OUTCOMES:
        y = pd.to_numeric(reg_model[outcome], errors="coerce")
        m = y.notna()
        xi = drop_collinear_columns(x_model.loc[m].copy())
        yi = y.loc[m].astype(float)

        if outcome in ELASTICITY_OUTCOMES:
            fit = sm.RLM(yi, xi, M=sm.robust.norms.HuberT()).fit(maxiter=200)
            pvals = 2.0 * norm.sf(np.abs(fit.params / fit.bse))
            pvals = pd.Series(pvals, index=fit.params.index)
            params = fit.params
            ses = fit.bse
        else:
            fit = sm.OLS(yi, xi).fit(cov_type="HC1")
            params = fit.params
            ses = fit.bse
            pvals = fit.pvalues

        for f in params.index:
            model_rows.append(
                {
                    "sample": "model_average_across_rules",
                    "outcome": outcome,
                    "feature": f,
                    "coef": float(params[f]),
                    "se": float(ses[f]),
                    "p_value": float(pvals[f]),
                    "interpretation": interpret_effect(outcome, float(params[f])),
                }
            )

    model_coef = pd.DataFrame(model_rows)
    model_sig = model_coef[
        (model_coef["feature"] != "const")
        & (model_coef["feature"] != "estimated")
        & (model_coef["p_value"] <= 0.10)
    ].copy()
    model_sig = model_sig.sort_values(["outcome", "p_value"]).reset_index(drop=True)

    return panel_coef, panel_sig, model_coef, model_sig




### rf_driver_importance


In [ ]:
def rf_driver_importance(reg):
    x = build_driver_features(reg)[DRIVER_ALL_FEATURES].copy()
    group_map = {
        "rules": DRIVER_CONTROL_FEATURES[:2],  # rule_itr, rule_g
        "nominal": DRIVER_NOMINAL_FEATURES,
        "real": DRIVER_REAL_FEATURES,
        "nonmodel": ["estimated"] + DRIVER_NONMODEL_FEATURES,
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rng = np.random.default_rng(42)

    feat_rows = []
    grp_rows = []

    for outcome in OUTCOMES:
        y = pd.to_numeric(reg[outcome], errors="coerce")
        m = y.notna()
        xi = x.loc[m].copy()
        yi = y.loc[m].to_numpy(dtype=float)

        feat_imp = np.zeros(xi.shape[1], dtype=float)
        grp_drop = {k: [] for k in group_map}
        base_scores = []

        for tr, te in cv.split(xi):
            xtr, xte = xi.iloc[tr].copy(), xi.iloc[te].copy()
            ytr, yte = yi[tr], yi[te]
            rf = RandomForestRegressor(
                n_estimators=250,
                random_state=42,
                min_samples_leaf=3,
                n_jobs=-1,
            )
            rf.fit(xtr, ytr)
            pred = rf.predict(xte)
            base_r2 = float(1.0 - np.sum((yte - pred) ** 2) / np.sum((yte - yte.mean()) ** 2)) if np.var(yte) > 0 else 0.0
            base_scores.append(base_r2)
            feat_imp += rf.feature_importances_

            for gname, cols in group_map.items():
                xp = xte.copy()
                idx = rng.permutation(len(xp))
                xp.loc[:, cols] = xp[cols].to_numpy()[idx]
                pred_p = rf.predict(xp)
                perm_r2 = float(1.0 - np.sum((yte - pred_p) ** 2) / np.sum((yte - yte.mean()) ** 2)) if np.var(yte) > 0 else 0.0
                grp_drop[gname].append(base_r2 - perm_r2)

        feat_imp = feat_imp / cv.get_n_splits()
        for f, imp in zip(xi.columns, feat_imp):
            feat_rows.append({"outcome": outcome, "feature": f, "rf_importance": float(imp)})

        for gname, vals in grp_drop.items():
            grp_rows.append(
                {
                    "outcome": outcome,
                    "group": gname,
                    "base_cv_r2": float(np.mean(base_scores)),
                    "group_permute_r2_drop": float(np.mean(vals)),
                }
            )

    feature_importance = pd.DataFrame(feat_rows).sort_values(["outcome", "rf_importance"], ascending=[True, False]).reset_index(drop=True)
    group_importance = pd.DataFrame(grp_rows).sort_values(["outcome", "group_permute_r2_drop"], ascending=[True, False]).reset_index(drop=True)
    return feature_importance, group_importance




### nonlinear_benchmark


In [ ]:
def nonlinear_benchmark(reg):
    x = reg[CORE_FEATURES].copy()
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    models = {
        "Ridge": Pipeline(
            [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]
        ),
        "RF": Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("m", RandomForestRegressor(n_estimators=120, random_state=42, min_samples_leaf=3, n_jobs=-1)),
            ]
        ),
        "HGB": Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                (
                    "m",
                    HistGradientBoostingRegressor(
                        random_state=42,
                        max_depth=4,
                        learning_rate=0.05,
                        max_iter=120,
                        min_samples_leaf=8,
                    ),
                ),
            ]
        ),
    }

    rows = []
    for outcome in OUTCOMES:
        y = reg[outcome]
        m = y.notna()
        xi = x.loc[m]
        yi = y.loc[m]
        for name, model in models.items():
            r2 = cross_val_score(model, xi, yi, cv=cv, scoring="r2")
            mae = -cross_val_score(model, xi, yi, cv=cv, scoring="neg_mean_absolute_error")
            rows.append(
                {
                    "outcome": outcome,
                    "model": name,
                    "n_obs": int(len(yi)),
                    "cv_r2_mean": float(r2.mean()),
                    "cv_r2_sd": float(r2.std()),
                    "cv_mae_mean": float(mae.mean()),
                }
            )
    return pd.DataFrame(rows).sort_values(["outcome", "model"]).reset_index(drop=True)




### outlier_sensitivity


In [ ]:
def outlier_sensitivity(reg):
    # 1/99-tail flag on each outcome; keep rows with no flags in trimmed sample.
    flag = pd.Series(False, index=reg.index)
    for outcome in OUTCOMES:
        q1, q99 = reg[outcome].quantile([0.01, 0.99])
        flag = flag | ((reg[outcome] < q1) | (reg[outcome] > q99))

    subsets = {"full": reg.copy(), "trimmed_1_99_any": reg.loc[~flag].copy()}
    models = {
        "Ridge": Pipeline(
            [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]
        ),
        "RF": Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("m", RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=3, n_jobs=-1)),
            ]
        ),
    }
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rows = []

    for subset_name, d in subsets.items():
        x = d[CORE_FEATURES]
        for outcome in OUTCOMES:
            y = d[outcome]
            m = y.notna()
            xi = x.loc[m]
            yi = y.loc[m]
            for model_name, model in models.items():
                r2 = cross_val_score(model, xi, yi, cv=cv, scoring="r2")
                rows.append(
                    {
                        "subset": subset_name,
                        "outcome": outcome,
                        "model": model_name,
                        "n_obs": int(len(yi)),
                        "cv_r2_mean": float(r2.mean()),
                    }
                )
    return pd.DataFrame(rows).sort_values(["outcome", "subset", "model"]).reset_index(drop=True)




### build_report


In [ ]:
def build_report(
    out_dir,
    var_decomp,
    rank_stability,
    rule_contrasts,
    panel_driver_sig,
    model_driver_sig,
    rf_feature_importance,
    rf_group_importance,
    cluster_outcomes,
    cluster_prevalence,
    cluster_stability_summary,
    cluster_stability_models,
    nonlinear,
    sensitivity,
    outliers,


## Build Outputs


### Step 1


In [ ]:
):
    ratio_df = var_decomp.copy()
    ratio_df["model_to_rule_increment_ratio"] = (
        ratio_df["inc_model_given_rule"] / ratio_df["inc_rule_given_model"].replace(0.0, np.nan)
    )

    switchers = cluster_stability_models[cluster_stability_models["stability_class"] == "rule_switcher"].copy()
    fast_cluster = int(cluster_outcomes.sort_values("y_timing_max").iloc[0]["cluster_raw"])
    slow_cluster = int(cluster_outcomes.sort_values("y_timing_max").iloc[-1]["cluster_raw"])
    fast_means = cluster_outcomes.loc[cluster_outcomes["cluster_raw"] == fast_cluster].iloc[0]
    slow_means = cluster_outcomes.loc[cluster_outcomes["cluster_raw"] == slow_cluster].iloc[0]

    panel_sig_short = panel_driver_sig[
        [
            "outcome",
            "feature",
            "coef",
            "p_value",
            "interpretation",
        ]
    ].copy()
    model_sig_short = model_driver_sig[
        [
            "outcome",
            "feature",
            "coef",
            "p_value",
            "interpretation",
        ]
    ].copy()

    rf_top = (
        rf_feature_importance.sort_values(["outcome", "rf_importance"], ascending=[True, False])
        .groupby("outcome")
        .head(8)
        .reset_index(drop=True)
    )

    lines = []
    lines.append("# Disentangle Spaghetti Report")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append("")
    lines.append("Identification used for driver analysis:")
    lines.append("- Main inference uses regressions with **rule controls** (`rule_itr`, `rule_g`) and model/non-model attributes as regressors.")
    lines.append("- We **do not** use model fixed effects to infer attribute effects, because model FE absorb time-invariant attributes.")
    lines.append("- Model FE analysis is included only as a diagnostic for where raw variance sits.")
    lines.append("")
    lines.append("Main conclusions:")
    lines.append(
        f"- Timing heterogeneity is driven most consistently by non-model features: `estimated` and `cb_authors_ext` are positive and significant in timing regressions; `cb_x_late` is negative for inflation timing."
    )
    lines.append(
        f"- Output-power heterogeneity is linked to specific structural interactions: `firm_bs_x_bank` is negative/significant for `IScurve20` (stronger output response), while `sw_x_noidx` is positive/significant (weaker output response)."
    )
    lines.append(
        f"- Inflation-power and disinflation-cost heterogeneity are tied to wage indexation structure: `sw_x_idx` raises `infl_per_rr20` (weaker inflation response) and raises `sacratio20` (higher disinflation cost), while `pr_ndx` lowers `sacratio20`."
    )
    lines.append(
        f"- Archetype structure is not noise: {int(cluster_stability_summary['n_models'].sum())} models split mostly into stable slow/moderate and fast/high-power classes with only {len(switchers)} rule-switchers."
    )
    lines.append("")

    lines.append("## 1) Diagnostic: Model vs Rule Variance Decomposition")
    lines.append("")
    lines.append(var_decomp.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Model-vs-rule dominance ratio (`inc_model_given_rule / inc_rule_given_model`):")
    lines.append(ratio_df[["outcome", "model_to_rule_increment_ratio"]].to_string(index=False, float_format=lambda v: f"{v:0.2f}"))
    lines.append("")
    lines.append("Interpretation: this section is diagnostic only; it does not identify attribute effects.")
    lines.append("")

    lines.append("## 2) Driver Regressions (Rule-Controlled, No Model FE)")
    lines.append("")
    lines.append("Significant attribute drivers (panel, p<=0.10; rules forced in and omitted from this list):")
    if panel_sig_short.empty:
        lines.append("(none)")
    else:
        lines.append(panel_sig_short.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Significant attribute drivers in model-averaged check (p<=0.10):")
    if model_sig_short.empty:
        lines.append("(none)")
    else:
        lines.append(model_sig_short.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")

    lines.append("## 3) Cross-Rule Rank Stability")
    lines.append("")
    lines.append(rank_stability.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Interpretation: model ranking persistence across rules is high (especially for timing and sacratio).")
    lines.append("")

    lines.append("## 4) Within-Model Rule Contrasts")
    lines.append("")
    lines.append(rule_contrasts.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Interpretation: rules shift levels/timing, but most models keep their relative ranking across rules.")
    lines.append("")

    lines.append("## 5) ML Cross-Check on Drivers")
    lines.append("")
    lines.append("Top random-forest feature importances by outcome (rule controls included):")
    lines.append(rf_top.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Grouped importance via group-permutation drop in holdout R2:")
    lines.append(rf_group_importance.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Interpretation: non-model features dominate timing predictability; rules dominate inflation-slope predictability.")
    lines.append("")

    lines.append("## 6) IRF Archetypes")
    lines.append("")
    lines.append("Cluster outcome means:")
    lines.append(cluster_outcomes.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Cluster attribute prevalence:")
    lines.append(cluster_prevalence.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Model-level cluster stability:")
    lines.append(cluster_stability_summary.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append(
        f"Fast/high-power archetype (cluster {fast_cluster}) mean y_timing={fast_means['y_timing_max']:.2f}, pi_timing={fast_means['piq_timing_max']:.2f}; "
        f"Slow/moderate archetype (cluster {slow_cluster}) mean y_timing={slow_means['y_timing_max']:.2f}, pi_timing={slow_means['piq_timing_max']:.2f}."
    )
    lines.append("")
    lines.append("Rule-switcher models:")
    if switchers.empty:
        lines.append("(none)")
    else:
        lines.append(switchers.to_string(index=False))
    lines.append("")

    lines.append("## 7) Nonlinear vs Linear Benchmark (CV)")
    lines.append("")
    lines.append(nonlinear.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")
    lines.append("Interpretation: nonlinear models materially improve fit for timing and sacratio, indicating interaction/nonlinearity in transmission differences.")
    lines.append("")

    lines.append("## 8) Outlier Sensitivity")
    lines.append("")
    lines.append("Extreme rows (1/99-tail on any outcome):")
    lines.append(outliers.to_string(index=False))
    lines.append("")
    lines.append("Predictive sensitivity (full vs trimmed):")
    lines.append(sensitivity.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    lines.append("")

    lines.append("## 9) So What")
    lines.append("")
    lines.append("- The main empirical split is **speed/persistence**, not just slope magnitudes.")
    lines.append("- `estimated`, `cb_authors_ext`, and vintage interactions matter most for timing outcomes.")
    lines.append("- Wage-indexation design is the clearest nominal-rigidity driver of inflation effectiveness/cost tradeoffs.")
    lines.append("- Real-friction interactions matter for output power but do not robustly map one-for-one into inflation power.")
    lines.append("- Therefore, “not much explains differences” is too pessimistic: differences are structured around a few recurring channels, with large residual heterogeneity still present.")
    lines.append("")

    (out_dir / "findings_report.md").write_text("\n".join(lines), encoding="utf-8")


out_dir = OUTPUT_DIR / "disentangle_spaghetti"
out_dir.mkdir(parents=True, exist_ok=True)

reg, irf = load_data()

print("Loaded regression rows:", len(reg))
print("Loaded IRF rows (<=20q):", len(irf))

var_decomp = variance_decomposition(reg)
rank_stability = rule_rank_stability(reg)
rule_contrasts = within_model_rule_contrasts(reg)
panel_driver_coef, panel_driver_sig, model_driver_coef, model_driver_sig = attribute_driver_models(reg)
rf_feature_importance, rf_group_importance = rf_driver_importance(reg)
outliers = outlier_table(reg)
shape_features, cluster_merged, cluster_outcomes, cluster_prevalence = cluster_archetypes(reg, irf)
cluster_stability_models, cluster_stability_summary = cluster_stability(shape_features, cluster_outcomes)
nonlinear = nonlinear_benchmark(reg)
sensitivity = outlier_sensitivity(reg)



### Step 2


In [ ]:
# Save raw outputs
var_decomp.to_csv(out_dir / "variance_decomposition.csv", index=False)
rank_stability.to_csv(out_dir / "rule_rank_stability.csv", index=False)
rule_contrasts.to_csv(out_dir / "within_model_rule_contrasts.csv", index=False)
panel_driver_coef.to_csv(out_dir / "attribute_driver_panel_coefficients.csv", index=False)
panel_driver_sig.to_csv(out_dir / "attribute_driver_panel_significant.csv", index=False)
model_driver_coef.to_csv(out_dir / "attribute_driver_modelavg_coefficients.csv", index=False)
model_driver_sig.to_csv(out_dir / "attribute_driver_modelavg_significant.csv", index=False)
rf_feature_importance.to_csv(out_dir / "rf_feature_importance.csv", index=False)
rf_group_importance.to_csv(out_dir / "rf_group_importance.csv", index=False)
outliers.to_csv(out_dir / "outlier_table_1_99.csv", index=False)
shape_features.to_csv(out_dir / "irf_shape_features.csv", index=False)
cluster_merged.to_csv(out_dir / "irf_archetype_assignments.csv", index=False)
cluster_outcomes.to_csv(out_dir / "irf_archetype_outcome_means.csv", index=False)
cluster_prevalence.to_csv(out_dir / "irf_archetype_attribute_prevalence.csv", index=False)
cluster_stability_models.to_csv(out_dir / "irf_archetype_model_stability.csv", index=False)
cluster_stability_summary.to_csv(out_dir / "irf_archetype_stability_summary.csv", index=False)
nonlinear.to_csv(out_dir / "nonlinear_benchmark_cv.csv", index=False)
sensitivity.to_csv(out_dir / "outlier_sensitivity_cv.csv", index=False)

build_report(
    out_dir=out_dir,
    var_decomp=var_decomp,
    rank_stability=rank_stability,
    rule_contrasts=rule_contrasts,
    panel_driver_sig=panel_driver_sig,
    model_driver_sig=model_driver_sig,
    rf_feature_importance=rf_feature_importance,
    rf_group_importance=rf_group_importance,
    cluster_outcomes=cluster_outcomes,
    cluster_prevalence=cluster_prevalence,
    cluster_stability_summary=cluster_stability_summary,
    cluster_stability_models=cluster_stability_models,
    nonlinear=nonlinear,
    sensitivity=sensitivity,
    outliers=outliers,
)

print("Wrote disentangle outputs to:", out_dir)
print(" -", out_dir / "findings_report.md")
print(" -", out_dir / "variance_decomposition.csv")
print(" -", out_dir / "rule_rank_stability.csv")
print(" -", out_dir / "within_model_rule_contrasts.csv")
print(" -", out_dir / "attribute_driver_panel_significant.csv")
print(" -", out_dir / "rf_group_importance.csv")
print(" -", out_dir / "irf_archetype_assignments.csv")
print(" -", out_dir / "irf_archetype_stability_summary.csv")
print(" -", out_dir / "nonlinear_benchmark_cv.csv")


